# Fase 3 — Data Preparation

**Em uma frase:** pegamos a planilha crua dos clientes e deixamos ela pronta para o modelo aprender.

Pense em um banco que vai treinar um analista novo. Você mostra o histórico de **80%** dos clientes (treino) e, no fim, testa se ele acerta nos **20%** que ele nunca viu (teste). Se você deixar o analista “espiar” os 20% enquanto estuda: por exemplo, calculando a renda típica com todo mundo misturado, a prova fica fácil demais.   
No jargão isso se chama **leakage** (vazamento): o teste deixa de ser um cliente novo.

O [01](01_business_understanding.ipynb) definiu o problema de negócio (prever calote grave em 2 anos). O [02](02_data_understanding.ipynb) achou os problemas na planilha (renda em branco, códigos 96/98, números absurdos). **Aqui aplicamos as regras** e separamos treino/teste. 

**Regra de ouro**

- Coisas que são **fato** (índice da planilha não é risco; idade 0 não existe; 96/98 não é “atrasou 98 vezes”) → consertamos **antes** de separar os grupos.
- Coisas que são **estatística da amostra** (renda típica, teto dos extremos) → calculamos **só no treino** e reutilizamos no teste. É o mesmo número que você usaria amanhã, no cliente que ainda não chegou.

**Ordem:** tirar o índice → consertar 96/98 → tirar idade 0 → separar treino/teste → marcar quem não informou renda → preencher buracos com números do treino → cortar extremos com teto do treino → salvar.

Este split (treino/teste com a mesma % de calote) é o conjunto em que os notebooks [04](04_modeling.ipynb) e [05](05_evaluation.ipynb) vão medir **AUC** e **Recall** definidos no [01](01_business_understanding.ipynb). Não calcule essas métricas aqui: isso exigiria treinar modelo e olharia o teste cedo demais.


## 1. Imports e `ROOT`

**O que esta célula faz (sem jargão):** diz ao computador *onde* está o projeto e *quais ferramentas* vamos usar. Não mexe em cliente nenhum ainda.

Este caderno quase sempre abre na pasta `notebooks/`. Se pedíssemos o arquivo como `data/raw/...`, o computador procuraria *dentro* de `notebooks/` e não acharia. `ROOT` é o “endereço da casa” — a pasta do repositório — para o caminho funcionar no seu Mac, no GitHub e no [04](04_modeling.ipynb).

Rode e confira: `ROOT` **não** pode terminar em `notebooks`.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split


def project_root() -> Path:
    """Sobe pastas até achar o CSV bruto — assim o caminho não depende do Mac de ninguém."""
    here = Path.cwd().resolve()
    for pasta in [here, *here.parents]:
        if (pasta / "data" / "raw" / "cs-training.csv").exists():
            return pasta
    raise FileNotFoundError("cs-training.csv não está em data/raw/. Veja o README.")


ROOT = project_root()
print("cwd :", Path.cwd())   # pasta onde o Jupyter abriu (geralmente notebooks/)
print("ROOT:", ROOT)         # pasta do projeto (tem que ser credit_risk_scoring)


## 2. Carregar o dado bruto

**O que esta célula faz:** abre a planilha original, do jeito que veio. Ainda **não** preenchemos buraco e **não** apagamos linha.

O que olhar no print:

- **150 mil linhas, 12 colunas** — cada linha é um cliente; uma das colunas é só o número da linha (`Unnamed: 0`), que não diz nada sobre risco.
- **PD ≈ 6,7%** — de cada 100 clientes, uns 7 tiveram atraso grave (90+ dias) nos 2 anos seguintes. Os outros ~93 não. Por isso “acertar quase todo mundo” dizendo “ninguém vai calotear” parece bom e é inútil (notebook 01).
- **Renda em branco em ~29,7 mil** (~20%) e dependentes em ~3,9 mil. Buraco na planilha **não** significa “cliente pior”: no 02, quem não informou renda tinha *menos* calote, não mais.

A **média** de utilização ou de dívida aqui não descreve o cliente típico: uns poucos números gigantes puxam a média para o espaço. Por isso mais tarde usamos **mediana** (o cliente do meio) e **teto P99** (corta só o 1% mais extremo).


In [ ]:
# Planilha original: ainda sem conserto. Conferir se é o arquivo certo.
df = pd.read_csv(ROOT / "data/raw/cs-training.csv")
print(df.shape)                       # quantos clientes × quantas colunas
print(df.columns.tolist())            # nomes das colunas
print(df["SeriousDlqin2yrs"].mean())  # % que deu calote grave (PD)
print(df.isna().sum())                # quantos buracos (NaN) em cada coluna


## 2b. Casos estranhos (contagens do [02](02_data_understanding.ipynb))

O 02 não limpou nada: só separou o que **não pode existir**, o que está **quebrado**, o que é **ambíguo** e o que é **extremo possível**. Abaixo, as mesmas contagens na planilha crua, e o que este notebook faz com cada uma.

A média de `RevolvingUtilizationOfUnsecuredLines` (~6) e de `DebtRatio` (~353) **não** descreve o cliente típico. Uns poucos números gigantes puxam a média; por isso mais à frente usamos mediana e teto Percentil 99.

`RevolvingUtilizationOfUnsecuredLines` é a fração do limite de crédito rotativo (cartão / linhas sem garantia) já em uso. **1 = 100% do limite.** Não confundir com `DebtRatio` (dívida / renda), que tem as linhas próprias na tabela.

| Caso | Quantos (base crua) | Tipo | O que este notebook faz |
|---|---|---|---|
| `Unnamed: 0` | 150 mil | Não é dado | Drop na seção 3 |
| `age == 0` | 1 (= único `< 18`) | Impossível | Drop (`age > 0`) na seção 3 |
| `age > 100` | 13 (P99 = 87) | Extremo possível | **Não mexe** (prioridade baixa) |
| 96/98 nas 3 colunas de atraso | 269, as **mesmas** linhas | Código / impossível | NaN na seção 3, depois 0 na seção 6 |
| `RevolvingUtilizationOfUnsecuredLines` `> 1` | 3.321 (~2%) | Ambíguo (over-limit pode ser real) | Não trata igual ao `> 10`; cap P99 na seção 7 |
| `RevolvingUtilizationOfUnsecuredLines` `> 10` | 241 | Quebrado (não é ratio de cartão) | Flag `util_gt_10` **antes** do cap, seção 7 |
| `DebtRatio > 1` | 35.137 (~23%) | Em geral real | **Não** apaga 35k linhas |
| `DebtRatio > 100` | 24.380 (~16%) | Quebrado / não-ratio | Dois tetos P99 pela **flag de renda**, seção 7 |
| Renda `NaN` | 29.731 (~19,8%) | Missing informativo (PD 5,61% vs 6,95%) | **Não dropa.** Flag + mediana do treino |
| Dependentes `NaN` | 3.924 (todos também sem renda) | Missing aninhado (PD 4,56%) | Flag + moda 0 do treino |
| Renda `== 0` | 1.634 | Ambíguo (desemprego ou missing disfarçado) | Mantém 0; imputa **só** NaN |
| Dependentes `> 10` | 2 | Extremo possível | **Não mexe** |
| Duplicados (sem ID) | ~609 linhas extra (~0,4%) | Qualidade | **Não mexe** (opcional, baixa prioridade) |

O achado que amarra vários “outliers”: `DebtRatio` alto e renda missing são o **mesmo** problema. Com renda, mediana do ratio ≈ 0,30 (só ~6% `> 1`). Sem renda, mediana ≈ 1.159 (~94% `> 1`) — provavelmente valor absoluto no lugar da fração. Por isso um P99 único na coluna inteira seria mentira.

Rode a célula seguinte e confira se os números batem com o 02.

In [ ]:
# Mesmas contagens do 02 (seção 1b). Ainda NÃO altera df.
util = "RevolvingUtilizationOfUnsecuredLines"
atrasos = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

print("age < 18:", (df["age"] < 18).sum())
print("age == 0:", (df["age"] == 0).sum())
print("age > 100:", (df["age"] > 100).sum())
print("RevolvingUtilizationOfUnsecuredLines > 1:", (df[util] > 1).sum())
print("RevolvingUtilizationOfUnsecuredLines > 10:", (df[util] > 10).sum())
print("DebtRatio > 1:", (df["DebtRatio"] > 1).sum())
print("DebtRatio > 100:", (df["DebtRatio"] > 100).sum())
for col in atrasos:
    print(f"{col} >= 96:", (df[col] >= 96).sum())
print(
    "linhas com >=96 nas TRÊS colunas:",
    ((df[atrasos[0]] >= 96) & (df[atrasos[1]] >= 96) & (df[atrasos[2]] >= 96)).sum(),
)
print("MonthlyIncome NaN:", df["MonthlyIncome"].isna().sum())
print("NumberOfDependents NaN:", df["NumberOfDependents"].isna().sum())
print("MonthlyIncome == 0:", (df["MonthlyIncome"] == 0).sum())
print("NumberOfDependents > 10:", (df["NumberOfDependents"] > 10).sum())
print(
    "duplicados (sem Unnamed: 0):",
    df.drop(columns=["Unnamed: 0"], errors="ignore").duplicated().sum(),
)

## 3. Regras que não dependem de quem caiu no treino

**O que esta célula faz:** conserta erros que seriam erros **em qualquer cliente**, hoje ou daqui a um ano. Por isso fazemos **antes** de separar treino e teste.

1. **Códigos 96/98 nas colunas de atraso.** O sistema às vezes grava 96 ou 98 quando “não sabe / não se aplica”. Não é “a pessoa atrasou 98 vezes” — em 2 anos isso é impossível. No 02 vimos que são **as mesmas 269 pessoas** nas três colunas. Trocamos por “buraco” (`NaN`) agora; o zero só entra **depois** da separação, para não misturar “código de sistema” com “nunca atrasou” cedo demais.
2. **`Unnamed: 0`.** É só a numeração da linha no Excel. Não é idade, não é renda, não é risco. Fora.
3. **Idade 0.** Um único cadastro impossível (o único menor de 18 da base). Tiramos essa linha. A idade mínima passa a ser 21.

Conferir: `(149999, 11)`, máximos de atraso **13 / 11 / 17** (sumiu o 96), `age.min() == 21`.


In [ ]:
# Três jeitos de contar atraso (30–59 dias, 60–89, 90+). 96/98 não são contagem real.
atrasos = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

# Conferência: no 02 eram 269 pessoas, as mesmas, nas três colunas.
for col in atrasos:
    print(f"{col} >= 96:", (df[col] >= 96).sum())
print(
    "linhas com >=96 nas TRÊS colunas:",
    ((df[atrasos[0]] >= 96) & (df[atrasos[1]] >= 96) & (df[atrasos[2]] >= 96)).sum(),
)

# Numeração da planilha não é risco.
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# Código especial → buraco (ainda não 0: 0 significa “nunca atrasou”).
for col in atrasos:
    df.loc[df[col] >= 96, col] = np.nan

# Um cadastro com idade 0: impossível. Fora.
df = df[df["age"] > 0].copy()

print(df.shape)
print("max atrasos:", [df[c].max() for c in atrasos])
print("age.min():", df["age"].min())


## 4. Separar treino e teste (a “prova”)

**Analogia:** o treino é a apostila; o teste é a prova com questões novas. Os **20%** do teste simulam o cliente que pede crédito **amanhã** — o modelo não pode ter estudado nele.

- `y` = o que queremos prever (deu calote grave? sim/não).
- `X` = o que sabemos hoje (idade, renda, atrasos passados…).
- **80% / 20%** — tamanho usual para ter bastante gente para aprender e um grupo honesto para avaliar.
- **`stratify=y`** — o calote é raro (~6,7%). Sem isso, o acaso poderia jogar 5% de caloteiros num lado e 8% no outro. Aí o “quanto pegamos de caloteiro” (Recall do notebook 01) seria sorte do corte, não qualidade do modelo. Com `stratify`, os dois lados ficam com a **mesma** taxa.
- **`random_state=42`** — o sorteio é repetível: amanhã o mesmo corte.


In [ ]:
y = df["SeriousDlqin2yrs"]           # o que queremos prever (1 = calote grave)
X = df.drop(columns=["SeriousDlqin2yrs"])  # o que sabemos hoje; o gabarito não entra aqui

# 80% estuda, 20% é prova. stratify = mesma % de calote dos dois lados.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print("colunas em X:", X.columns.tolist())
print("target em X?", "SeriousDlqin2yrs" in X.columns)  # tem que ser False
print("Unnamed: 0 em X?", "Unnamed: 0" in X.columns)     # tem que ser False
print("PD treino:", y_train.mean())  # ~6,68% — igualzinho ao teste
print("PD teste:", y_test.mean())


## 5. Bandeirinhas **antes** de preencher o buraco

**Por que isso importa no banco:** “não declarou renda” e “ganha 5.400” são histórias diferentes. Se preenchermos o buraco primeiro com 5.400, os dois viram o **mesmo número** e o modelo não consegue mais distinguir.

A **flag** (bandeirinha 0/1) é um post-it: “esta pessoa não informou renda”. Guardamos isso **agora**, enquanto o buraco ainda está visível.

No 02, quem não informou renda tinha **menos** calote (cerca de 5,6% vs 7,0%). Missing **não** é “mais risco”. Por isso também **não** jogamos fora os ~20% sem renda: é um quinto da carteira e um perfil real de quem pede crédito.

O mesmo vale para dependentes em branco. A célula abaixo imprime a PD dos dois grupos: no 02, quem não informou dependentes também caloteava **menos**.


In [ ]:
# Cópia para não alterar o recorte original por acidente.
X_train = X_train.copy()
X_test = X_test.copy()

# Post-it 0/1: 1 = não informou. Tem que nascer ANTES de preencher o buraco.
X_train["income_was_missing"] = X_train["MonthlyIncome"].isna().astype(int)
X_test["income_was_missing"] = X_test["MonthlyIncome"].isna().astype(int)
X_train["dependents_was_missing"] = X_train["NumberOfDependents"].isna().astype(int)
X_test["dependents_was_missing"] = X_test["NumberOfDependents"].isna().astype(int)

print("flag renda (treino) mean:", X_train["income_was_missing"].mean())  # ~20%
# Calote no grupo sem renda tem que ser MENOR (não é “buraco = pior cliente”).
print("PD treino, renda missing:", y_train[X_train["income_was_missing"] == 1].mean())
print("PD treino, renda informada:", y_train[X_train["income_was_missing"] == 0].mean())
print("flag dependentes (treino) mean:", X_train["dependents_was_missing"].mean())
print("PD treino, dependentes missing:", y_train[X_train["dependents_was_missing"] == 1].mean())
print("PD treino, dependentes informados:", y_train[X_train["dependents_was_missing"] == 0].mean())


## 6. Preencher buracos (com a “régua” do treino)

**Analogia:** medimos a renda típica só nos clientes do treino. Amanhã, um cliente novo sem renda recebe **essa mesma** régua — não recalculamos com ele dentro.

- **Mediana da renda (não a média):** alinhe todo mundo por renda; o do **meio** é a mediana. A média é puxada pelos muito ricos (aqui ~6.659 vs mediana **5.400**).
- **Moda dos dependentes:** o valor que mais aparece — em geral **0** (sem dependentes).
- **Renda igual a 0** já foi informada (“ganho zero”). Não trocamos por 5.400.
- **Atrasos que viraram buraco** (os antigos 96/98) agora viram **0**: “não temos evidência de atraso real”. Não usamos a média de atrasos, porque a média misturaria o código de erro com comportamento de verdade.

O teste usa **os números do treino**, mesmo que a mediana *dele* fosse 5.450.


In [ ]:
# Quem já declarou renda 0 continua 0 — não é buraco.
n_renda_zero_antes = (X_train["MonthlyIncome"] == 0).sum()

# Régua só no treino. O teste (e o cliente de amanhã) reutiliza estes números.
mediana_renda = X_train["MonthlyIncome"].median()  # cliente do meio, não a média dos ricos
moda_dep = X_train["NumberOfDependents"].mode()[0]  # valor que mais aparece (em geral 0)
print("mediana_renda:", mediana_renda)
print("moda_dep:", moda_dep)
print("média renda (treino, só não-nulos):", X_train["MonthlyIncome"].mean())  # deve ser > mediana

for parte in (X_train, X_test):
    parte["MonthlyIncome"] = parte["MonthlyIncome"].fillna(mediana_renda)
    parte["NumberOfDependents"] = parte["NumberOfDependents"].fillna(moda_dep)
    for col in atrasos:
        parte[col] = parte[col].fillna(0)  # 96/98 já viraram NaN; agora “sem atraso observado”

print("isna treino:\n", X_train.isna().sum())  # tudo zero daqui pra frente
print("isna teste:\n", X_test.isna().sum())
print("MonthlyIncome == 0 (treino) antes:", n_renda_zero_antes)
print("MonthlyIncome == 0 (treino) depois:", (X_train["MonthlyIncome"] == 0).sum())  # igual ao antes


## 7. Teto nos extremos (P99 do treino)

**O que é um cap / P99:** a linha do cliente **fica**. Só o valor absurdo vira um teto. “Percentil 99” = o número que só 1% das pessoas do treino passa. Acima disso, cortamos — senão um único cadastro com utilização 50.000 distorce o aprendizado.

Três casos diferentes (não misturar):

1. **`RevolvingUtilizationOfUnsecuredLines`.** Usar um pouco acima de 100% (`> 1`) pode ser real (estourou o limite). Já `> 10` (1.000%) não é ratio de cartão: é dado quebrado. Por isso uma bandeirinha `util_gt_10` **antes** do corte, e o teto no P99 (~1,10).
2. **Dívida / renda (`DebtRatio`).** Quem informou renda tem um mundo (fração, mediana ~0,30). Quem **não** informou tem outro (número enorme, como se a dívida viesse crua). Um teto só para a coluna inteira seria mentira. Dois tetos, pela **bandeirinha de renda** — não por “renda == 5400”, que mistura quem realmente ganha isso com quem tinha buraco.
3. **Renda.** Também ganha um teto no P99 do treino, para supermilionários não puxarem o modelo.

No fim: **13 colunas** = 10 originais + 3 bandeirinhas (renda em branco, dependentes em branco, utilização quebrada).


In [ ]:
util = "RevolvingUtilizationOfUnsecuredLines"

# > 1 pode ser over-limit real; > 10 é cadastro quebrado. Não são o mesmo problema.
print("RevolvingUtilizationOfUnsecuredLines > 1 (treino):", (X_train[util] > 1).sum())
print("RevolvingUtilizationOfUnsecuredLines > 10 (treino):", (X_train[util] > 10).sum())

# Bandeirinha ANTES do teto: depois do clip, ninguém fica > 10 e a informação se perderia.
X_train["util_gt_10"] = (X_train[util] > 10).astype(int)
X_test["util_gt_10"] = (X_test[util] > 10).astype(int)

# Teto = P99 do treino (só 1% passa disso). O teste usa o mesmo teto.
p99_util = X_train[util].quantile(0.99)
print("p99_util:", p99_util)
X_train[util] = X_train[util].clip(upper=p99_util)
X_test[util] = X_test[util].clip(upper=p99_util)
print("RevolvingUtilizationOfUnsecuredLines > 10 depois do clip (treino):", (X_train[util] > 10).sum())  # zero

# Dois mundos de DebtRatio: fração (com renda) vs número enorme (sem renda).
mask_ok = X_train["income_was_missing"] == 0
mask_na = X_train["income_was_missing"] == 1
p99_debt_ok = X_train.loc[mask_ok, "DebtRatio"].quantile(0.99)
p99_debt_na = X_train.loc[mask_na, "DebtRatio"].quantile(0.99)
print("P99 DebtRatio renda informada:", p99_debt_ok)
print("P99 DebtRatio renda missing:", p99_debt_na)

for parte in (X_train, X_test):
    ok = parte["income_was_missing"] == 0
    na = parte["income_was_missing"] == 1
    parte.loc[ok, "DebtRatio"] = parte.loc[ok, "DebtRatio"].clip(upper=p99_debt_ok)
    parte.loc[na, "DebtRatio"] = parte.loc[na, "DebtRatio"].clip(upper=p99_debt_na)

p99_renda = X_train["MonthlyIncome"].quantile(0.99)
print("p99_renda:", p99_renda)
X_train["MonthlyIncome"] = X_train["MonthlyIncome"].clip(upper=p99_renda)
X_test["MonthlyIncome"] = X_test["MonthlyIncome"].clip(upper=p99_renda)

print("n colunas treino:", X_train.shape[1])  # 13 = 10 originais + 3 flags
print("colunas:", X_train.columns.tolist())


## 8. Guardar a planilha pronta

**O que esta célula faz:** grava quatro arquivos para o [04](04_modeling.ipynb) (modelagem) ler. O 04 **não** deve reabrir o CSV cru nem refazer o preenchimento — senão alguém recalcula a mediana “olhando” o teste.

- `index=False` evita nascer outro `Unnamed: 0` (aquela coluna de numeração inútil).
- O endereço tem que ser a pasta do **projeto** (`…/credit_risk_scoring/data/processed`), não `notebooks/data/processed`.
- Conferir: **119.999 × 13** no treino e **30.000 × 13** no teste; PD de `y` relida ~6,68%; o path **não** contém `notebooks`.
- `impute_params.json` guarda a régua do treino (mediana, moda, P99) para o cliente de amanhã — o 06 pode reaplicar sem recalcular no teste.

O que **não** fazemos hoje: treinar modelo, SMOTE, pesos de classe, scaler. Isso é a Fase 4.


In [ ]:
import json

# Pasta do projeto (não notebooks/). O 04 só lê daqui.
out = ROOT / "data/processed"
out.mkdir(parents=True, exist_ok=True)
X_train.to_csv(out / "X_train.csv", index=False)  # index=False: não recriar Unnamed: 0
X_test.to_csv(out / "X_test.csv", index=False)
y_train.to_csv(out / "y_train.csv", index=False)
y_test.to_csv(out / "y_test.csv", index=False)

# Régua do treino: o cliente de amanhã usa estes números, não recalcula.
params = {
    "mediana_renda": float(mediana_renda),
    "moda_dep": float(moda_dep),
    "p99_util": float(p99_util),
    "p99_debt_ok": float(p99_debt_ok),
    "p99_debt_na": float(p99_debt_na),
    "p99_renda": float(p99_renda),
}
(out / "impute_params.json").write_text(json.dumps(params, indent=2))
print(out.resolve())
print("path tem notebooks/?", "notebooks" in str(out.resolve()))
print("régua:", params)

# Relê para conferir o contrato do 04: 13 colunas, mesma PD.
xtr = pd.read_csv(out / "X_train.csv")
xte = pd.read_csv(out / "X_test.csv")
ytr = pd.read_csv(out / "y_train.csv").squeeze()
yte = pd.read_csv(out / "y_test.csv").squeeze()
print(xtr.shape, xte.shape)
print(ytr.shape, yte.shape)
print("PD y_train relido:", float(ytr.mean()))
print("PD y_test relido:", float(yte.mean()))
print(xtr.columns.tolist())


## Minhas notas (em linguagem de negócio)

Respostas com os números **desta** execução.

1. Por que a mediana de renda é só do treino?

Porque a mediana é a “régua” de quem já conhecemos. Se eu medir a renda típica misturando treino e teste, a prova já viu a apostila — cola. Em produção o cliente de amanhã não entra nessa conta. Aqui a régua do treino foi **5.400**; o teste usou o mesmo 5.400, mesmo que a mediana *dele* fosse outra.

2. Por que a bandeirinha nasce antes de preencher o buraco?

Depois de colocar 5.400 no lugar vazio, “não declarou renda” e “ganha 5.400” viram a mesma pessoa na planilha. O post-it `income_was_missing` guarda a diferença que o 02 achou: no treino ~**19,8%** sem renda, com calote **5,55%** vs **6,96%** entre quem informou. Se a bandeirinha ficasse tudo 0, o preenchimento veio na ordem errada.

3. Por que não jogar fora os 20% sem renda?

São **29.731** clientes — um quinto da carteira. Não é um buraco aleatório: eles caloteiam **menos**, não mais, e o campo de dívida nesse grupo não é uma fração de renda. Apagar seria fingir que esse público não pede crédito. O scoring do 01 tem que valer também para quem chega sem preencher renda.

4. Por que manter a mesma % de calote nos dois lados (`stratify`)?

Calote é raro (~**6,68%**). Sem esse cuidado, o acaso poderia colocar 5% de caloteiros num lado e 8% no outro. Aí “quantos caloteiros pegamos” (Recall) seria sorte do sorteio, não qualidade do modelo. Com o corte repetível: **6,684%** no treino e **6,683%** no teste (**119.999** vs **30.000** pessoas).

5. Por que o save usa `ROOT` e não a pasta “daqui do lado”?

O caderno abre em `notebooks/`. Um caminho curto criaria a pasta no lugar errado. `ROOT` é o endereço da casa do projeto; o [04](04_modeling.ipynb) procura os arquivos **lá**, não em `notebooks/data/processed`.

6. Por que dois tetos de dívida/renda?

Quem informou renda tem um ratio (teto **655**). Quem não informou tem um número enorme (teto **8.138**) — provavelmente dívida crua, não fração. Um teto só misturaria os dois mundos. O corte segue a **bandeirinha**, não o valor 5.400.

7. Por que 96/98 viram 0 e não a média de atrasos?

As **269** pessoas têm 96/98 nas **três** colunas: é código de sistema, não “atrasou 98 vezes” em 2 anos. Viraram buraco e depois **0** (“sem atraso observado”). A média misturaria erro de cadastro com comportamento real. Depois do conserto o máximo ficou **13 / 11 / 17**.

**Números desta run:** bruto 150.000 × 12, PD 6,684%, renda em branco 29.731; após regras 149.999 × 11, idade mínima 21; split 119.999 / 30.000; mediana renda 5.400, moda dependentes 0, média de renda ~6.659; renda 0 no treino 1.299 antes e depois; teto utilização ~1,10; tetos DebtRatio 655 vs 8.138; teto renda ~23.018; arquivos finais 119.999 × 13 e 30.000 × 13.

**Onde salvou:** `/Users/vinicius/Documents/workspace/credit_risk_scoring/data/processed`
